In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip uninstall -y datasets huggingface_hub -q
!pip install -q "datasets==3.6.0" "huggingface_hub==0.34.4" pandas numpy pyarrow scikit-learn tqdm sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.8 MB/s eta 0:00:00


In [3]:
import datasets
import huggingface_hub

print("datasets version:", datasets.__version__)
print("huggingface_hub version:", huggingface_hub.__version__)

datasets version: 3.6.0
huggingface_hub version: 0.34.4


In [4]:
import os
import re
import json
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [5]:
PROJECT_DIR = Path("/content/drive/MyDrive/MXH FINAL")

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
CACHE_DIR = PROJECT_DIR / "hf_cache"

for folder in [PROJECT_DIR, RAW_DIR, SILVER_DIR, GOLD_DIR, CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Project folder: /content/drive/MyDrive/MXH FINAL


In [6]:
#CATEGORY = "Gift_Cards"
CATEGORY = "All_Beauty"

DATASET_NAME = "McAuley-Lab/Amazon-Reviews-2023"

# Folder Google Drive của bạn
PROJECT_DIR = Path("/content/drive/MyDrive/MXH FINAL")

BASE_DIR = PROJECT_DIR / "data"
RAW_DIR = BASE_DIR / "raw"
SILVER_DIR = BASE_DIR / "silver"
GOLD_DIR = BASE_DIR / "gold"
CACHE_DIR = PROJECT_DIR / "hf_cache"

for folder in [PROJECT_DIR, RAW_DIR, SILVER_DIR, GOLD_DIR, CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Nếu chạy All_Beauty mà máy yếu, đặt SAMPLE_SIZE = 100000
# Nếu muốn chạy toàn bộ thì để SAMPLE_SIZE = None
SAMPLE_SIZE = None
# SAMPLE_SIZE = 100000

# Bước embedding khá lâu, ban đầu nên để False
GENERATE_EMBEDDINGS = False
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

RANDOM_STATE = 42

print("PROJECT_DIR:", PROJECT_DIR)
print("GOLD_DIR:", GOLD_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("CATEGORY:", CATEGORY)

PROJECT_DIR: /content/drive/MyDrive/MXH FINAL
GOLD_DIR: /content/drive/MyDrive/MXH FINAL/data/gold
CACHE_DIR: /content/drive/MyDrive/MXH FINAL/hf_cache
CATEGORY: All_Beauty


In [7]:
def load_hf_config_to_pandas(dataset_name: str, config_name: str, cache_dir: Path) -> pd.DataFrame:
    try:
        ds = load_dataset(
            dataset_name,
            config_name,
            split="full",
            trust_remote_code=True,
            cache_dir=str(cache_dir)
        )
        return ds.to_pandas()

    except Exception as e:
        print("Không load được split='full'. Thử load DatasetDict...")
        print("Lỗi ban đầu:", e)

        ds_dict = load_dataset(
            dataset_name,
            config_name,
            trust_remote_code=True,
            cache_dir=str(cache_dir)
        )

        first_split = list(ds_dict.keys())[0]
        print("Dùng split:", first_split)

        return ds_dict[first_split].to_pandas()


def load_amazon_reviews(category: str) -> pd.DataFrame:
    config_name = f"raw_review_{category}"
    return load_hf_config_to_pandas(DATASET_NAME, config_name, CACHE_DIR)


def load_amazon_metadata(category: str) -> pd.DataFrame:
    config_name = f"raw_meta_{category}"
    return load_hf_config_to_pandas(DATASET_NAME, config_name, CACHE_DIR)

# Nếu muốn lấy sample để chạy nhanh
if SAMPLE_SIZE is not None and len(reviews_raw) > SAMPLE_SIZE:
    reviews_raw = reviews_raw.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print("After sampling:", reviews_raw.shape)

In [8]:
reviews_raw = load_amazon_reviews(CATEGORY)

print("Raw reviews shape:", reviews_raw.shape)
print("Columns:", reviews_raw.columns.tolist())

reviews_raw.head()

README.md: 0.00B [00:00, ?B/s]

Amazon-Reviews-2023.py: 0.00B [00:00, ?B/s]

raw/review_categories/All_Beauty.jsonl:   0%|          | 0.00/327M [00:00<?, ?B/s]

Generating full split: 0 examples [00:00, ? examples/s]

Raw reviews shape: (701528, 10)
Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True
3,1.0,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True
4,5.0,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True


In [9]:
LOAD_METADATA = False

if LOAD_METADATA:
    metadata_raw = load_amazon_metadata(CATEGORY)
    print("Raw metadata shape:", metadata_raw.shape)
    print("Metadata columns:", metadata_raw.columns.tolist())
    display(metadata_raw.head())
else:
    metadata_raw = None
    print("Bỏ qua metadata để pipeline nhẹ hơn.")

Bỏ qua metadata để pipeline nhẹ hơn.


In [10]:
def convert_timestamp(series: pd.Series) -> pd.Series:
    """
    Amazon timestamp thường là Unix timestamp dạng millisecond.
    Hàm này xử lý linh hoạt cả second và millisecond.
    """
    s = pd.to_numeric(series, errors="coerce")

    if s.dropna().empty:
        return pd.to_datetime(series, errors="coerce")

    median_value = s.dropna().median()

    if median_value > 10**12:
        return pd.to_datetime(s, unit="ms", errors="coerce")
    else:
        return pd.to_datetime(s, unit="s", errors="coerce")


def make_review_id(row) -> str:
    """
    Tạo review_id ổn định từ user_id, item_id, timestamp, rating và review_text.
    """
    raw = f"{row['user_id']}|{row['item_id']}|{row['timestamp']}|{row['rating']}|{row['review_text']}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


def standardize_reviews(df: pd.DataFrame) -> pd.DataFrame:
    """
    Chuẩn hóa review data về schema chung:
    review_id, user_id, item_id, rating, review_title, review_text,
    timestamp, helpful_vote, verified_purchase.
    """
    df = df.copy()

    if "parent_asin" in df.columns:
        df["item_id"] = df["parent_asin"]
    elif "asin" in df.columns:
        df["item_id"] = df["asin"]
    else:
        raise ValueError("Không tìm thấy cột asin hoặc parent_asin.")

    if "title" in df.columns:
        df["review_title"] = df["title"].fillna("").astype(str)
    else:
        df["review_title"] = ""

    if "text" in df.columns:
        df["review_text"] = df["text"].fillna("").astype(str)
    else:
        raise ValueError("Không tìm thấy cột text.")

    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    if "timestamp" in df.columns:
        df["timestamp"] = convert_timestamp(df["timestamp"])
    else:
        raise ValueError("Không tìm thấy cột timestamp.")

    if "helpful_vote" in df.columns:
        df["helpful_vote"] = pd.to_numeric(df["helpful_vote"], errors="coerce").fillna(0)
    else:
        df["helpful_vote"] = 0

    if "verified_purchase" in df.columns:
        df["verified_purchase"] = df["verified_purchase"].fillna(False).astype(bool)
    else:
        df["verified_purchase"] = False

    keep_cols = [
        "user_id",
        "item_id",
        "rating",
        "review_title",
        "review_text",
        "timestamp",
        "helpful_vote",
        "verified_purchase"
    ]

    df = df[keep_cols]

    return df

reviews = standardize_reviews(reviews_raw)

print("Standardized shape:", reviews.shape)
reviews.head()

Standardized shape: (701528, 8)


,user_id,item_id,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,2020-05-05 14:08:48.923,0,True
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",2020-05-04 18:10:55.070,1,True
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,Yes!,"Smells good, feels great!",2020-05-16 21:41:06.052,2,True
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,1.0,Synthetic feeling,Felt synthetic,2022-01-28 18:13:50.220,0,True
4,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,5.0,A+,Love it,2020-12-30 10:02:43.534,0,True


In [11]:
def clean_reviews(df: pd.DataFrame, min_words: int = 3) -> pd.DataFrame:
    """
    Làm sạch dữ liệu review:
    - Xóa missing critical fields
    - Xóa review quá ngắn
    - Xóa rating ngoài 1-5
    - Xóa duplicate
    - Tạo review_id
    """
    df = df.copy()

    required_cols = ["user_id", "item_id", "rating", "review_text", "timestamp"]
    df = df.dropna(subset=required_cols)

    df["user_id"] = df["user_id"].astype(str).str.strip()
    df["item_id"] = df["item_id"].astype(str).str.strip()
    df["review_text"] = df["review_text"].astype(str).str.strip()
    df["review_title"] = df["review_title"].astype(str).str.strip()

    df = df[df["user_id"] != ""]
    df = df[df["item_id"] != ""]
    df = df[df["review_text"] != ""]

    df = df[df["rating"].between(1, 5)]

    df["word_count_temp"] = df["review_text"].str.split().str.len()
    df = df[df["word_count_temp"] >= min_words]
    df = df.drop(columns=["word_count_temp"])

    df = df.drop_duplicates(
        subset=["user_id", "item_id", "timestamp", "rating", "review_text"]
    )

    df["review_id"] = df.apply(make_review_id, axis=1)

    cols = [
        "review_id",
        "user_id",
        "item_id",
        "rating",
        "review_title",
        "review_text",
        "timestamp",
        "helpful_vote",
        "verified_purchase"
    ]

    df = df[cols].reset_index(drop=True)

    return df

reviews_clean = clean_reviews(reviews)

print("Clean reviews shape:", reviews_clean.shape)
reviews_clean.head()

Clean reviews shape: (644689, 9)


,review_id,user_id,item_id,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase
0,0242d5a54252829ae530f89370285f62,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,2020-05-05 14:08:48.923,0,True
1,c3bf884c1c7e73f0b8b2d3860e533e6f,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",2020-05-04 18:10:55.070,1,True
2,0751846bddcfc0c8e0f80616b5adbeaa,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,Yes!,"Smells good, feels great!",2020-05-16 21:41:06.052,2,True
3,240f069d52740d5c715863c6fbf0ca1a,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,Pretty Color,The polish was quiet thick and did not apply s...,2020-08-27 22:30:08.138,0,True
4,084eee9ececbb86725f2430052abd8aa,AHREXOGQPZDA6354MHH4ETSF3MCQ,B099DRHW5V,5.0,Handy,Great for many tasks. I purchased these for m...,2021-09-17 13:31:59.443,0,True


In [12]:
def normalize_text_for_tfidf(text: str) -> str:
    """
    Làm sạch nhẹ cho TF-IDF hoặc thống kê text.
    Không dùng cột này cho transformer embedding.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s!?.,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def add_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["text_raw"] = (
        df["review_title"].fillna("").astype(str).str.strip()
        + " "
        + df["review_text"].fillna("").astype(str).str.strip()
    ).str.strip()

    df["text_clean"] = df["text_raw"].apply(normalize_text_for_tfidf)

    return df

In [13]:
reviews_clean = add_text_columns(reviews_clean)

reviews_clean[["review_id", "text_raw", "text_clean"]].head()

,review_id,text_raw,text_clean
0,0242d5a54252829ae530f89370285f62,Such a lovely scent but not overpowering. This...,such a lovely scent but not overpowering. this...
1,c3bf884c1c7e73f0b8b2d3860e533e6f,Works great but smells a little weird. This pr...,works great but smells a little weird. this pr...
2,0751846bddcfc0c8e0f80616b5adbeaa,"Yes! Smells good, feels great!","yes! smells good, feels great!"
3,240f069d52740d5c715863c6fbf0ca1a,Pretty Color The polish was quiet thick and di...,pretty color the polish was quiet thick and di...
4,084eee9ececbb86725f2430052abd8aa,Handy Great for many tasks. I purchased these...,handy great for many tasks. i purchased these ...


In [14]:
def count_uppercase_words(text: str) -> int:
    words = str(text).split()
    return sum(1 for w in words if len(w) > 1 and w.isupper())


def add_text_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["text_length_chars"] = df["text_raw"].str.len()
    df["text_length_words"] = df["text_raw"].str.split().str.len()
    df["num_exclamation"] = df["text_raw"].str.count("!")
    df["num_question"] = df["text_raw"].str.count(r"\?")
    df["num_uppercase_words"] = df["text_raw"].apply(count_uppercase_words)

    df["uppercase_word_ratio"] = (
        df["num_uppercase_words"] / df["text_length_words"].replace(0, np.nan)
    ).fillna(0)

    df["is_short_review"] = (df["text_length_words"] <= 8).astype(int)
    df["is_very_short_review"] = (df["text_length_words"] <= 4).astype(int)

    df["text_fingerprint"] = (
        df["text_clean"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    df["duplicate_text_count"] = df.groupby("text_fingerprint")["review_id"].transform("count")

    return df

In [15]:
reviews_clean = add_text_features(reviews_clean)

reviews_clean.head()

,review_id,user_id,item_id,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase,text_raw,...,text_length_chars,text_length_words,num_exclamation,num_question,num_uppercase_words,uppercase_word_ratio,is_short_review,is_very_short_review,text_fingerprint,duplicate_text_count
0,0242d5a54252829ae530f89370285f62,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,2020-05-05 14:08:48.923,0,True,Such a lovely scent but not overpowering. This...,...,342,68,1,0,0,0.0,0,0,such a lovely scent but not overpowering. this...,1
1,c3bf884c1c7e73f0b8b2d3860e533e6f,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",2020-05-04 18:10:55.070,1,True,Works great but smells a little weird. This pr...,...,274,54,0,0,0,0.0,0,0,works great but smells a little weird. this pr...,1
2,0751846bddcfc0c8e0f80616b5adbeaa,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,Yes!,"Smells good, feels great!",2020-05-16 21:41:06.052,2,True,"Yes! Smells good, feels great!",...,30,5,2,0,0,0.0,1,0,"yes! smells good, feels great!",1
3,240f069d52740d5c715863c6fbf0ca1a,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,Pretty Color,The polish was quiet thick and did not apply s...,2020-08-27 22:30:08.138,0,True,Pretty Color The polish was quiet thick and di...,...,138,26,0,0,0,0.0,0,0,pretty color the polish was quiet thick and di...,1
4,084eee9ececbb86725f2430052abd8aa,AHREXOGQPZDA6354MHH4ETSF3MCQ,B099DRHW5V,5.0,Handy,Great for many tasks. I purchased these for m...,2021-09-17 13:31:59.443,0,True,Handy Great for many tasks. I purchased these...,...,150,23,0,0,0,0.0,0,0,handy great for many tasks. i purchased these ...,1


In [16]:
def add_user_features(df: pd.DataFrame):
    df = df.copy()

    user_agg = df.groupby("user_id").agg(
        user_review_count=("review_id", "count"),
        user_item_count=("item_id", "nunique"),
        user_avg_rating=("rating", "mean"),
        user_rating_std=("rating", "std"),
        user_first_review_time=("timestamp", "min"),
        user_last_review_time=("timestamp", "max"),
        user_verified_ratio=("verified_purchase", "mean"),
        user_avg_text_length=("text_length_words", "mean"),
    ).reset_index()

    user_agg["user_rating_std"] = user_agg["user_rating_std"].fillna(0)

    user_agg["user_active_days"] = (
        user_agg["user_last_review_time"] - user_agg["user_first_review_time"]
    ).dt.days + 1

    user_agg["user_reviews_per_day"] = (
        user_agg["user_review_count"] / user_agg["user_active_days"].replace(0, 1)
    )

    user_agg["user_singleton"] = (user_agg["user_review_count"] == 1).astype(int)

    df = df.merge(user_agg, on="user_id", how="left")

    return df, user_agg

In [17]:
reviews_clean, users_clean = add_user_features(reviews_clean)

print("Users shape:", users_clean.shape)
users_clean.head()

Users shape: (588332, 12)


,user_id,user_review_count,user_item_count,user_avg_rating,user_rating_std,user_first_review_time,user_last_review_time,user_verified_ratio,user_avg_text_length,user_active_days,user_reviews_per_day,user_singleton
0,AE222BBOVZIF42YOOPNBXL4UUMYA,1,1,5.0,0.0,2016-03-10 00:27:52.000,2016-03-10 00:27:52.000,1.0,8.0,1,1.0,1
1,AE222X475JC6ONXMIKZDFGQ7IAUA,1,1,5.0,0.0,2017-01-06 18:55:24.000,2017-01-06 18:55:24.000,1.0,4.0,1,1.0,1
2,AE222Y4WTST6BUZ4J5Y2H6QMBITQ,1,1,4.0,0.0,2013-06-24 21:11:42.000,2013-06-24 21:11:42.000,1.0,34.0,1,1.0,1
3,AE2232TEZOEWQLAFEX2NA6VBGMYQ,1,1,5.0,0.0,2019-07-30 05:47:26.197,2019-07-30 05:47:26.197,1.0,7.0,1,1.0,1
4,AE22355IAQGYY4EPWWVYX25J7AHA,1,1,1.0,0.0,2013-05-31 00:47:30.000,2013-05-31 00:47:30.000,1.0,32.0,1,1.0,1


In [18]:
def add_item_features(df: pd.DataFrame):
    df = df.copy()

    item_agg = df.groupby("item_id").agg(
        item_review_count=("review_id", "count"),
        item_user_count=("user_id", "nunique"),
        item_avg_rating=("rating", "mean"),
        item_rating_std=("rating", "std"),
        item_first_review_time=("timestamp", "min"),
        item_last_review_time=("timestamp", "max"),
        item_verified_ratio=("verified_purchase", "mean"),
        item_avg_text_length=("text_length_words", "mean"),
    ).reset_index()

    item_agg["item_rating_std"] = item_agg["item_rating_std"].fillna(0)

    item_agg["item_active_days"] = (
        item_agg["item_last_review_time"] - item_agg["item_first_review_time"]
    ).dt.days + 1

    item_agg["item_reviews_per_day"] = (
        item_agg["item_review_count"] / item_agg["item_active_days"].replace(0, 1)
    )

    df = df.merge(item_agg, on="item_id", how="left")

    return df, item_agg

In [19]:
reviews_clean, items_clean = add_item_features(reviews_clean)

print("Items shape:", items_clean.shape)
items_clean.head()

Items shape: (108784, 11)


,item_id,item_review_count,item_user_count,item_avg_rating,item_rating_std,item_first_review_time,item_last_review_time,item_verified_ratio,item_avg_text_length,item_active_days,item_reviews_per_day
0,0124784577,3,3,4.333333,1.154701,2019-06-24 23:29:59.244,2019-09-09 14:59:29.877,1.000000,17.666667,77,0.038961
1,0515059560,1,1,4.000000,0.000000,2014-09-17 02:00:11.000,2014-09-17 02:00:11.000,1.000000,47.000000,1,1.000000
2,0692508988,1,1,5.000000,0.000000,2016-03-24 21:27:06.000,2016-03-24 21:27:06.000,1.000000,50.000000,1,1.000000
3,069267599X,39,39,4.794872,0.656124,2016-10-21 16:55:20.000,2022-01-24 15:42:09.966,0.974359,43.871795,1921,0.020302
4,0764490117,2,2,5.000000,0.000000,2013-06-21 21:07:30.000,2015-09-01 19:48:49.000,0.500000,78.000000,802,0.002494


In [20]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["review_date"] = df["timestamp"].dt.date.astype(str)
    df["review_year"] = df["timestamp"].dt.year
    df["review_month"] = df["timestamp"].dt.month
    df["review_year_month"] = df["timestamp"].dt.to_period("M").astype(str)
    df["review_dayofweek"] = df["timestamp"].dt.dayofweek

    df["user_day_review_count"] = df.groupby(["user_id", "review_date"])["review_id"].transform("count")

    df["item_day_review_count"] = df.groupby(["item_id", "review_date"])["review_id"].transform("count")

    df["item_month_review_count"] = df.groupby(["item_id", "review_year_month"])["review_id"].transform("count")

    df["item_rating_month_count"] = df.groupby(
        ["item_id", "rating", "review_year_month"]
    )["review_id"].transform("count")

    return df

In [21]:
reviews_clean = add_temporal_features(reviews_clean)

reviews_clean.head()

,review_id,user_id,item_id,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase,text_raw,...,item_reviews_per_day,review_date,review_year,review_month,review_year_month,review_dayofweek,user_day_review_count,item_day_review_count,item_month_review_count,item_rating_month_count
0,0242d5a54252829ae530f89370285f62,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,2020-05-05 14:08:48.923,0,True,Such a lovely scent but not overpowering. This...,...,0.035604,2020-05-05,2020,5,2020-05,1,1,1,1,1
1,c3bf884c1c7e73f0b8b2d3860e533e6f,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",2020-05-04 18:10:55.070,1,True,Works great but smells a little weird. This pr...,...,0.048327,2020-05-04,2020,5,2020-05,0,1,1,4,1
2,0751846bddcfc0c8e0f80616b5adbeaa,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,Yes!,"Smells good, feels great!",2020-05-16 21:41:06.052,2,True,"Yes! Smells good, feels great!",...,0.017516,2020-05-16,2020,5,2020-05,5,1,1,1,1
3,240f069d52740d5c715863c6fbf0ca1a,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,Pretty Color,The polish was quiet thick and did not apply s...,2020-08-27 22:30:08.138,0,True,Pretty Color The polish was quiet thick and di...,...,0.005225,2020-08-27,2020,8,2020-08,3,1,1,1,1
4,084eee9ececbb86725f2430052abd8aa,AHREXOGQPZDA6354MHH4ETSF3MCQ,B099DRHW5V,5.0,Handy,Great for many tasks. I purchased these for m...,2021-09-17 13:31:59.443,0,True,Handy Great for many tasks. I purchased these...,...,0.285714,2021-09-17,2021,9,2021-09,4,1,1,2,2


In [22]:
def add_rating_deviation_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["rating_deviation_from_item_avg"] = (
        df["rating"] - df["item_avg_rating"]
    ).abs()

    df["rating_deviation_from_user_avg"] = (
        df["rating"] - df["user_avg_rating"]
    ).abs()

    df["is_extreme_rating"] = df["rating"].isin([1, 5]).astype(int)

    return df

In [23]:
reviews_clean = add_rating_deviation_features(reviews_clean)

reviews_clean.head()

,review_id,user_id,item_id,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase,text_raw,...,review_month,review_year_month,review_dayofweek,user_day_review_count,item_day_review_count,item_month_review_count,item_rating_month_count,rating_deviation_from_item_avg,rating_deviation_from_user_avg,is_extreme_rating
0,0242d5a54252829ae530f89370285f62,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,2020-05-05 14:08:48.923,0,True,Such a lovely scent but not overpowering. This...,...,5,2020-05,1,1,1,1,1,1.217391,0.5,1
1,c3bf884c1c7e73f0b8b2d3860e533e6f,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",2020-05-04 18:10:55.070,1,True,Works great but smells a little weird. This pr...,...,5,2020-05,0,1,1,4,1,0.076923,0.5,0
2,0751846bddcfc0c8e0f80616b5adbeaa,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,Yes!,"Smells good, feels great!",2020-05-16 21:41:06.052,2,True,"Yes! Smells good, feels great!",...,5,2020-05,5,1,1,1,1,1.454545,0.0,1
3,240f069d52740d5c715863c6fbf0ca1a,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,Pretty Color,The polish was quiet thick and did not apply s...,2020-08-27 22:30:08.138,0,True,Pretty Color The polish was quiet thick and di...,...,8,2020-08,3,1,1,1,1,1.000000,0.0,0
4,084eee9ececbb86725f2430052abd8aa,AHREXOGQPZDA6354MHH4ETSF3MCQ,B099DRHW5V,5.0,Handy,Great for many tasks. I purchased these for m...,2021-09-17 13:31:59.443,0,True,Handy Great for many tasks. I purchased these...,...,9,2021-09,4,1,1,2,2,0.000000,0.0,1


In [24]:
def add_weak_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tạo suspicion_score và weak_label.

    weak_label = 1 nghĩa là review có dấu hiệu bất thường.
    Đây KHÔNG phải nhãn fake thật.
    """
    df = df.copy()

    score = np.zeros(len(df))

    # Rule 1: Review ngắn nhưng rating cực đoan
    score += ((df["is_short_review"] == 1) & (df["is_extreme_rating"] == 1)).astype(int)

    # Rule 2: Text bị lặp lại nhiều lần
    score += (df["duplicate_text_count"] >= 3).astype(int)

    # Rule 3: User viết nhiều review trong cùng ngày
    score += (df["user_day_review_count"] >= 3).astype(int)

    # Rule 4: Item nhận nhiều review trong cùng ngày
    score += (df["item_day_review_count"] >= 5).astype(int)

    # Rule 5: Item có nhiều review cùng rating trong cùng tháng
    score += (df["item_rating_month_count"] >= 5).astype(int)

    # Rule 6: Không verified purchase và rating cực đoan
    score += ((df["verified_purchase"] == False) & (df["is_extreme_rating"] == 1)).astype(int)

    # Rule 7: User hoạt động rất ngắn nhưng có nhiều review
    score += ((df["user_active_days"] <= 3) & (df["user_review_count"] >= 3)).astype(int)

    # Rule 8: Helpful vote bằng 0, review ngắn, rating cực đoan
    score += (
        (df["helpful_vote"] == 0)
        & (df["is_short_review"] == 1)
        & (df["is_extreme_rating"] == 1)
    ).astype(int)

    df["suspicion_score"] = score

    # Có thể chỉnh ngưỡng này sau khi xem phân phối
    df["weak_label"] = (df["suspicion_score"] >= 4).astype(int)

    return df

In [25]:
reviews_clean = add_weak_labels(reviews_clean)

print("Weak label distribution:")
print(reviews_clean["weak_label"].value_counts())

print("\nSuspicion score description:")
print(reviews_clean["suspicion_score"].describe())

reviews_clean[[
    "review_id",
    "user_id",
    "item_id",
    "rating",
    "text_raw",
    "suspicion_score",
    "weak_label"
]].head(10)

Weak label distribution:
weak_label
0    643245
1      1444
Name: count, dtype: int64

Suspicion score description:
count    644689.000000
mean          0.397917
std           0.741571
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           6.000000
Name: suspicion_score, dtype: float64


,review_id,user_id,item_id,rating,text_raw,suspicion_score,weak_label
0,0242d5a54252829ae530f89370285f62,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,Such a lovely scent but not overpowering. This...,0.0,0
1,c3bf884c1c7e73f0b8b2d3860e533e6f,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,Works great but smells a little weird. This pr...,0.0,0
2,0751846bddcfc0c8e0f80616b5adbeaa,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,"Yes! Smells good, feels great!",1.0,0
3,240f069d52740d5c715863c6fbf0ca1a,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,Pretty Color The polish was quiet thick and di...,0.0,0
4,084eee9ececbb86725f2430052abd8aa,AHREXOGQPZDA6354MHH4ETSF3MCQ,B099DRHW5V,5.0,Handy Great for many tasks. I purchased these...,0.0,0
5,0033f17204acf8dc3ee692b083e69321,AEYORY2AVPMCPDV57CE337YU5LXA,B08BBQ29N5,3.0,Meh These were lightweight and soft but much t...,0.0,0
6,517ba6ef628c15411733643c4121a460,AFSKPY37N3C43SOI5IEXEK5JSIYA,B08P2DZB4X,5.0,Great for at home use and so easy to use! This...,1.0,0
7,566ee217272d5aca74a7495347dd67cb,AFSKPY37N3C43SOI5IEXEK5JSIYA,B086QY6T7N,5.0,Nice shampoo for the money I get Keratin treat...,1.0,0
8,d2dfc65b0fa4b7f83ca13d46567016bc,AFSKPY37N3C43SOI5IEXEK5JSIYA,B08DHTJ25J,3.0,Not what I thought I would be getting I was ve...,0.0,0
9,87f7ef44ef16c65e036bc9f423a35ea3,AFSKPY37N3C43SOI5IEXEK5JSIYA,B07RBSLNFR,5.0,A little goes a long way! This is a really nic...,1.0,0


In [26]:
def add_time_based_split(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop=True)

    n = len(df)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    df["split"] = "test"
    df.loc[:train_end - 1, "split"] = "train"
    df.loc[train_end:val_end - 1, "split"] = "val"

    return df

In [27]:
reviews_clean = add_time_based_split(reviews_clean)

print(reviews_clean["split"].value_counts())

reviews_clean[["review_id", "timestamp", "weak_label", "split"]].head()

split
train    451282
test      96704
val       96703
Name: count, dtype: int64


,review_id,timestamp,weak_label,split
0,0113f88a1bf0b2c280be0f7b51b082d8,2000-11-01 04:24:18,0,train
1,f2bfb7e145c554057a8343ee4546c727,2001-01-16 15:10:44,0,train
2,ef62107bfa3da60898c98d0a6df9b2ab,2001-01-30 17:13:34,0,train
3,07a3ca4d1b961d36be63e0d5a6039ec0,2001-03-05 07:27:57,0,train
4,3a998ffe0a711b9542b02808b406be50,2001-03-29 23:38:47,0,train


In [28]:
def build_node_tables(df: pd.DataFrame):
    """
    Tạo 3 loại node:
    - User
    - Review
    - Item/Product
    """
    nodes_review = df[[
        "review_id",
        "user_id",
        "item_id",
        "rating",
        "timestamp",
        "text_raw",
        "text_clean",
        "helpful_vote",
        "verified_purchase",
        "suspicion_score",
        "weak_label",
        "split"
    ]].drop_duplicates("review_id").reset_index(drop=True)

    nodes_user = df[[
        "user_id",
        "user_review_count",
        "user_item_count",
        "user_avg_rating",
        "user_rating_std",
        "user_active_days",
        "user_reviews_per_day",
        "user_verified_ratio",
        "user_avg_text_length",
        "user_singleton"
    ]].drop_duplicates("user_id").reset_index(drop=True)

    nodes_item = df[[
        "item_id",
        "item_review_count",
        "item_user_count",
        "item_avg_rating",
        "item_rating_std",
        "item_active_days",
        "item_reviews_per_day",
        "item_verified_ratio",
        "item_avg_text_length"
    ]].drop_duplicates("item_id").reset_index(drop=True)

    return nodes_user, nodes_review, nodes_item

In [29]:
nodes_user, nodes_review, nodes_item = build_node_tables(reviews_clean)

print("User nodes:", nodes_user.shape)
print("Review nodes:", nodes_review.shape)
print("Item nodes:", nodes_item.shape)

nodes_review.head()

User nodes: (588332, 10)
Review nodes: (644689, 12)
Item nodes: (108784, 9)


,review_id,user_id,item_id,rating,timestamp,text_raw,text_clean,helpful_vote,verified_purchase,suspicion_score,weak_label,split
0,0113f88a1bf0b2c280be0f7b51b082d8,AED2GFGIAJ22PHMZGSKH2CPUF75Q,B000050FDE,5.0,2000-11-01 04:24:18,"The best electric toothbrush ever, REALLY! We ...","the best electric toothbrush ever, really! we ...",10,False,1.0,0,train
1,f2bfb7e145c554057a8343ee4546c727,AH54X3UMWTAMUJU2CVWYWNZVETLA,B000050FDE,2.0,2001-01-16 15:10:44,Fine while it's working I paid the full... for...,fine while it s working i paid the full... for...,20,False,0.0,0,train
2,ef62107bfa3da60898c98d0a6df9b2ab,AHQGZITP7IEYTISSUELRAKFRH3GA,B000050AUD,5.0,2001-01-30 17:13:34,Over [price] for a toothbrush?? It's worth eve...,over price for a toothbrush?? it s worth every...,24,False,1.0,0,train
3,07a3ca4d1b961d36be63e0d5a6039ec0,AEG7T4QNZ2EZEE4QVV6WZB2LXCOQ,B000050AUD,5.0,2001-03-05 07:27:57,"Why did I wait so long? I admit it, I put off ...","why did i wait so long? i admit it, i put off ...",7,False,1.0,0,train
4,3a998ffe0a711b9542b02808b406be50,AHC4HE4WV6CUPI4K74KQ7YCPRPWA,B000050AUD,5.0,2001-03-29 23:38:47,I wouldn't be without it....... I purchased th...,i wouldn t be without it....... i purchased th...,4,False,1.0,0,train


In [30]:
def create_consecutive_edges(
    df: pd.DataFrame,
    group_cols: list,
    src_col: str = "review_id",
    time_col: str = "timestamp",
    edge_type: str = "relation"
) -> pd.DataFrame:
    """
    Tạo edge bằng cách nối các review liền kề trong cùng group.
    Cách này nhẹ hơn tạo clique toàn phần.
    """
    edges = []

    grouped = df.sort_values(time_col).groupby(group_cols)

    for _, group in tqdm(grouped, desc=f"Building {edge_type} edges"):
        ids = group[src_col].tolist()

        if len(ids) < 2:
            continue

        for i in range(len(ids) - 1):
            edges.append({
                "src": ids[i],
                "dst": ids[i + 1],
                "edge_type": edge_type
            })

    return pd.DataFrame(edges)


def build_edge_tables(df: pd.DataFrame):
    """
    Tạo các edge chính:
    - User writes Review
    - Review about Item
    - User rates Item
    - Review-Review suspicious/behavioral relations
    """
    edges_user_review = df[[
        "user_id",
        "review_id",
        "timestamp"
    ]].copy()

    edges_user_review = edges_user_review.rename(columns={
        "user_id": "src",
        "review_id": "dst"
    })
    edges_user_review["edge_type"] = "user_writes_review"

    edges_review_item = df[[
        "review_id",
        "item_id",
        "timestamp"
    ]].copy()

    edges_review_item = edges_review_item.rename(columns={
        "review_id": "src",
        "item_id": "dst"
    })
    edges_review_item["edge_type"] = "review_about_item"

    edges_user_item = df[[
        "user_id",
        "item_id",
        "rating",
        "timestamp",
        "verified_purchase"
    ]].copy()

    edges_user_item = edges_user_item.rename(columns={
        "user_id": "src",
        "item_id": "dst"
    })
    edges_user_item["edge_type"] = "user_rates_item"

    edges_same_user = create_consecutive_edges(
        df,
        group_cols=["user_id"],
        edge_type="same_user"
    )

    edges_same_item_rating_month = create_consecutive_edges(
        df,
        group_cols=["item_id", "rating", "review_year_month"],
        edge_type="same_item_same_rating_same_month"
    )

    edges_same_item_day = create_consecutive_edges(
        df,
        group_cols=["item_id", "review_date"],
        edge_type="same_item_same_day"
    )

    edges_review_review = pd.concat(
        [
            edges_same_user,
            edges_same_item_rating_month,
            edges_same_item_day
        ],
        ignore_index=True
    )

    if len(edges_review_review) > 0:
        edges_review_review = edges_review_review.drop_duplicates(
            subset=["src", "dst", "edge_type"]
        ).reset_index(drop=True)
    else:
        edges_review_review = pd.DataFrame(columns=["src", "dst", "edge_type"])

    return {
        "edges_user_review": edges_user_review,
        "edges_review_item": edges_review_item,
        "edges_user_item": edges_user_item,
        "edges_review_review": edges_review_review
    }

In [31]:
edge_tables = build_edge_tables(reviews_clean)

for name, edge_df in edge_tables.items():
    print(name, edge_df.shape)

edge_tables["edges_review_review"].head()

Building same_user edges:   0%|          | 0/588332 [00:00<?, ?it/s]

Building same_item_same_rating_same_month edges:   0%|          | 0/472074 [00:00<?, ?it/s]

Building same_item_same_day edges:   0%|          | 0/605824 [00:00<?, ?it/s]

edges_user_review (644689, 4)
edges_review_item (644689, 4)
edges_user_item (644689, 6)
edges_review_review (267837, 3)


,src,dst,edge_type
0,85e2505bc9b89c698e28ddc703743ea0,c62b91463d6dea25ee79519ff62ab352,same_user
1,8fc91cfdfae8e583763f731336d7be3c,1cb1e7be61c97d7e69e6b8a335ede5f9,same_user
2,1cb1e7be61c97d7e69e6b8a335ede5f9,6cf859250b6c8308c19e2beb6d15be2e,same_user
3,b0a13c86625872881c03634bb3418ee3,3dc4e7d2664e63aaf81fd5cb0010b5ba,same_user
4,3292516792e508681fe528acb382dbb4,0e2b863e649b04f509adcdc8e331359f,same_user


In [32]:
def build_review_feature_table(df: pd.DataFrame) -> pd.DataFrame:
    feature_cols = [
        "review_id",

        # Text features
        "text_length_chars",
        "text_length_words",
        "num_exclamation",
        "num_question",
        "num_uppercase_words",
        "uppercase_word_ratio",
        "is_short_review",
        "is_very_short_review",
        "duplicate_text_count",

        # Rating features
        "rating",
        "is_extreme_rating",
        "rating_deviation_from_item_avg",
        "rating_deviation_from_user_avg",

        # Review metadata
        "helpful_vote",
        "verified_purchase",

        # User behavior
        "user_review_count",
        "user_item_count",
        "user_avg_rating",
        "user_rating_std",
        "user_active_days",
        "user_reviews_per_day",
        "user_verified_ratio",
        "user_avg_text_length",
        "user_singleton",
        "user_day_review_count",

        # Item behavior
        "item_review_count",
        "item_user_count",
        "item_avg_rating",
        "item_rating_std",
        "item_active_days",
        "item_reviews_per_day",
        "item_verified_ratio",
        "item_avg_text_length",
        "item_day_review_count",
        "item_month_review_count",
        "item_rating_month_count",

        # Weak label
        "suspicion_score",
        "weak_label"
    ]

    features = df[feature_cols].copy()

    features["verified_purchase"] = features["verified_purchase"].astype(int)

    return features

In [33]:
review_features = build_review_feature_table(reviews_clean)

print("Review features shape:", review_features.shape)
review_features.head()

Review features shape: (644689, 39)


,review_id,text_length_chars,text_length_words,num_exclamation,num_question,num_uppercase_words,uppercase_word_ratio,is_short_review,is_very_short_review,duplicate_text_count,...,item_rating_std,item_active_days,item_reviews_per_day,item_verified_ratio,item_avg_text_length,item_day_review_count,item_month_review_count,item_rating_month_count,suspicion_score,weak_label
0,0113f88a1bf0b2c280be0f7b51b082d8,282,45,2,0,1,0.022222,0,0,1,...,0.808287,7550,0.025166,0.789474,89.273684,1,1,1,1.0,0
1,f2bfb7e145c554057a8343ee4546c727,356,63,0,0,0,0.000000,0,0,1,...,0.808287,7550,0.025166,0.789474,89.273684,1,1,1,0.0,0
2,ef62107bfa3da60898c98d0a6df9b2ab,2669,493,10,2,3,0.006085,0,0,1,...,1.468569,5401,0.004999,0.185185,172.259259,1,1,1,1.0,0
3,07a3ca4d1b961d36be63e0d5a6039ec0,1406,267,1,4,1,0.003745,0,0,1,...,1.468569,5401,0.004999,0.185185,172.259259,1,2,2,1.0,0
4,3a998ffe0a711b9542b02808b406be50,1115,211,0,0,0,0.000000,0,0,1,...,1.468569,5401,0.004999,0.185185,172.259259,1,2,2,1.0,0


In [34]:
GENERATE_EMBEDDINGS = True

In [35]:
def generate_text_embeddings(
    df: pd.DataFrame,
    text_col: str = "text_raw",
    model_name: str = EMBEDDING_MODEL_NAME,
    batch_size: int = 64,
    output_path: Path = None
):
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(model_name)

    texts = df[text_col].fillna("").astype(str).tolist()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    embeddings = np.array(embeddings)

    if output_path is not None:
        np.save(output_path, embeddings)

    return embeddings

In [36]:
if GENERATE_EMBEDDINGS:
    embedding_path = GOLD_DIR / CATEGORY / f"review_text_embeddings_{CATEGORY}.npy"
    embedding_path.parent.mkdir(parents=True, exist_ok=True)

    review_embeddings = generate_text_embeddings(
        nodes_review,
        text_col="text_raw",
        model_name=EMBEDDING_MODEL_NAME,
        batch_size=64,
        output_path=embedding_path
    )

    print("Embedding shape:", review_embeddings.shape)
    print("Saved to:", embedding_path)
else:
    print("Skip embedding generation. Set GENERATE_EMBEDDINGS = True to run.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10074 [00:00<?, ?it/s]

Embedding shape: (644689, 384)
Saved to: /content/drive/MyDrive/MXH FINAL/data/gold/All_Beauty/review_text_embeddings_All_Beauty.npy


In [37]:
def save_pipeline_outputs(
    category: str,
    reviews_clean: pd.DataFrame,
    nodes_user: pd.DataFrame,
    nodes_review: pd.DataFrame,
    nodes_item: pd.DataFrame,
    edge_tables: dict,
    review_features: pd.DataFrame
):
    output_dir = GOLD_DIR / category
    output_dir.mkdir(parents=True, exist_ok=True)

    reviews_clean.to_parquet(output_dir / "reviews_clean.parquet", index=False)
    nodes_user.to_parquet(output_dir / "nodes_user.parquet", index=False)
    nodes_review.to_parquet(output_dir / "nodes_review.parquet", index=False)
    nodes_item.to_parquet(output_dir / "nodes_item.parquet", index=False)

    for name, edge_df in edge_tables.items():
        edge_df.to_parquet(output_dir / f"{name}.parquet", index=False)

    review_features.to_parquet(output_dir / "review_features.parquet", index=False)

    review_splits = reviews_clean[[
        "review_id",
        "split",
        "weak_label",
        "suspicion_score"
    ]].copy()

    review_splits.to_parquet(output_dir / "review_splits.parquet", index=False)

    summary = {
        "category": category,
        "num_reviews": int(len(nodes_review)),
        "num_users": int(len(nodes_user)),
        "num_items": int(len(nodes_item)),
        "num_edges_user_review": int(len(edge_tables["edges_user_review"])),
        "num_edges_review_item": int(len(edge_tables["edges_review_item"])),
        "num_edges_user_item": int(len(edge_tables["edges_user_item"])),
        "num_edges_review_review": int(len(edge_tables["edges_review_review"])),
        "weak_label_distribution": {
            str(k): int(v) for k, v in reviews_clean["weak_label"].value_counts().to_dict().items()
        },
        "suspicion_score_distribution": {
            str(k): int(v) for k, v in reviews_clean["suspicion_score"].value_counts().sort_index().to_dict().items()
        },
        "split_distribution": {
            str(k): int(v) for k, v in reviews_clean["split"].value_counts().to_dict().items()
        }
    }

    with open(output_dir / "pipeline_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4, ensure_ascii=False)

    return output_dir, summary

In [38]:
output_dir, summary = save_pipeline_outputs(
    category=CATEGORY,
    reviews_clean=reviews_clean,
    nodes_user=nodes_user,
    nodes_review=nodes_review,
    nodes_item=nodes_item,
    edge_tables=edge_tables,
    review_features=review_features
)

print("Saved to:", output_dir)
summary

Saved to: /content/drive/MyDrive/MXH FINAL/data/gold/All_Beauty


{'category': 'All_Beauty',
 'num_reviews': 644689,
 'num_users': 588332,
 'num_items': 108784,
 'num_edges_user_review': 644689,
 'num_edges_review_item': 644689,
 'num_edges_user_item': 644689,
 'num_edges_review_review': 267837,
 'weak_label_distribution': {'0': 643245, '1': 1444},
 'suspicion_score_distribution': {'0.0': 471541,
  '1.0': 104690,
  '2.0': 55114,
  '3.0': 11900,
  '4.0': 1310,
  '5.0': 129,
  '6.0': 5},
 'split_distribution': {'train': 451282, 'test': 96704, 'val': 96703}}

In [39]:
print("Output files:")

for file in output_dir.iterdir():
    print("-", file.name)

Output files:
- review_text_embeddings_All_Beauty.npy
- reviews_clean.parquet
- nodes_user.parquet
- nodes_review.parquet
- nodes_item.parquet
- edges_user_review.parquet
- edges_review_item.parquet
- edges_user_item.parquet
- edges_review_review.parquet
- review_features.parquet
- review_splits.parquet
- pipeline_summary.json


In [40]:
pd.read_parquet(output_dir / "reviews_clean.parquet").head()

,review_id,user_id,item_id,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase,text_raw,...,user_day_review_count,item_day_review_count,item_month_review_count,item_rating_month_count,rating_deviation_from_item_avg,rating_deviation_from_user_avg,is_extreme_rating,suspicion_score,weak_label,split
0,0113f88a1bf0b2c280be0f7b51b082d8,AED2GFGIAJ22PHMZGSKH2CPUF75Q,B000050FDE,5.0,"The best electric toothbrush ever, REALLY!",We have used Oral-B products for 15 years; thi...,2000-11-01 04:24:18,10,False,"The best electric toothbrush ever, REALLY! We ...",...,1,1,1,1,0.489474,0.0,1,1.0,0,train
1,f2bfb7e145c554057a8343ee4546c727,AH54X3UMWTAMUJU2CVWYWNZVETLA,B000050FDE,2.0,Fine while it's working,"I paid the full... for mine, but had to return...",2001-01-16 15:10:44,20,False,Fine while it's working I paid the full... for...,...,1,1,1,1,2.510526,0.0,0,0.0,0,train
2,ef62107bfa3da60898c98d0a6df9b2ab,AHQGZITP7IEYTISSUELRAKFRH3GA,B000050AUD,5.0,Over [price] for a toothbrush?? It's worth eve...,"First of all, I am not a dental professional.....",2001-01-30 17:13:34,24,False,Over [price] for a toothbrush?? It's worth eve...,...,1,1,1,1,0.814815,0.0,1,1.0,0,train
3,07a3ca4d1b961d36be63e0d5a6039ec0,AEG7T4QNZ2EZEE4QVV6WZB2LXCOQ,B000050AUD,5.0,Why did I wait so long?,"I admit it, I put off buying the Sonicare beca...",2001-03-05 07:27:57,7,False,"Why did I wait so long? I admit it, I put off ...",...,1,1,2,2,0.814815,0.0,1,1.0,0,train
4,3a998ffe0a711b9542b02808b406be50,AHC4HE4WV6CUPI4K74KQ7YCPRPWA,B000050AUD,5.0,I wouldn't be without it.......,I purchased this tooth brush about five years ...,2001-03-29 23:38:47,4,False,I wouldn't be without it....... I purchased th...,...,1,1,2,2,0.814815,0.0,1,1.0,0,train


In [41]:
pd.read_parquet(output_dir / "review_features.parquet").head()

,review_id,text_length_chars,text_length_words,num_exclamation,num_question,num_uppercase_words,uppercase_word_ratio,is_short_review,is_very_short_review,duplicate_text_count,...,item_rating_std,item_active_days,item_reviews_per_day,item_verified_ratio,item_avg_text_length,item_day_review_count,item_month_review_count,item_rating_month_count,suspicion_score,weak_label
0,0113f88a1bf0b2c280be0f7b51b082d8,282,45,2,0,1,0.022222,0,0,1,...,0.808287,7550,0.025166,0.789474,89.273684,1,1,1,1.0,0
1,f2bfb7e145c554057a8343ee4546c727,356,63,0,0,0,0.000000,0,0,1,...,0.808287,7550,0.025166,0.789474,89.273684,1,1,1,0.0,0
2,ef62107bfa3da60898c98d0a6df9b2ab,2669,493,10,2,3,0.006085,0,0,1,...,1.468569,5401,0.004999,0.185185,172.259259,1,1,1,1.0,0
3,07a3ca4d1b961d36be63e0d5a6039ec0,1406,267,1,4,1,0.003745,0,0,1,...,1.468569,5401,0.004999,0.185185,172.259259,1,2,2,1.0,0
4,3a998ffe0a711b9542b02808b406be50,1115,211,0,0,0,0.000000,0,0,1,...,1.468569,5401,0.004999,0.185185,172.259259,1,2,2,1.0,0


In [42]:
pd.read_parquet(output_dir / "edges_review_review.parquet").head()

,src,dst,edge_type
0,85e2505bc9b89c698e28ddc703743ea0,c62b91463d6dea25ee79519ff62ab352,same_user
1,8fc91cfdfae8e583763f731336d7be3c,1cb1e7be61c97d7e69e6b8a335ede5f9,same_user
2,1cb1e7be61c97d7e69e6b8a335ede5f9,6cf859250b6c8308c19e2beb6d15be2e,same_user
3,b0a13c86625872881c03634bb3418ee3,3dc4e7d2664e63aaf81fd5cb0010b5ba,same_user
4,3292516792e508681fe528acb382dbb4,0e2b863e649b04f509adcdc8e331359f,same_user
